# Notebook 2: Traditional Baselines (Lead-3 & TextRank)
Thực thi trên Kaggle: Cài đặt các thư viện cần thiết
!pip install pandas datasets evaluate rouge_score sumy nltk bert_score sacrebleu

In [ ]:
import pandas as pd
import nltk
from evaluate import load
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
import numpy as np
import warnings
warnings.filterwarnings("ignore")

nltk.download('punkt')

In [ ]:
# Tải tập test đã lưu từ Notebook 1
print("Loading Test set...")
test_df = pd.read_csv("test_1k.csv") # Nhớ mount Dataset từ output của NB1 trên Kaggle
articles = test_df["article"].tolist()
references = test_df["abstract"].tolist()

# Tải metric ROUGE, BLEU, BERTScore
rouge = load("rouge")
bleu = load("bleu")
bertscore = load("bertscore")

def evaluate_summaries(predictions, refs):
    r_score = rouge.compute(predictions=predictions, references=refs)
    bleu_score = bleu.compute(predictions=predictions, references=refs)
    bert_result = bertscore.compute(predictions=predictions, references=refs, lang="vi")
    
    return {
        "ROUGE-1": round(r_score['rouge1'] * 100, 2), 
        "ROUGE-2": round(r_score['rouge2'] * 100, 2),
        "ROUGE-L": round(r_score['rougeL'] * 100, 2),
        "ROUGE-Lsum": round(r_score['rougeLsum'] * 100, 2),
        "BLEU": round(bleu_score['bleu'] * 100, 2),
        "BERT-F1": round(np.mean(bert_result['f1']) * 100, 2)
    }

## 1. Lead-3 Baseline
Lấy 3 câu đầu tiên của bài báo làm tóm tắt.

In [ ]:
def lead_3_summarize(text):
    sentences = nltk.sent_tokenize(text)
    return " ".join(sentences[:3])

print("Running Lead-3...")
lead3_preds = [lead_3_summarize(article) for article in articles]
lead3_results = evaluate_summaries(lead3_preds, references)
print("=== Lead-3 Metrics ===")
print(lead3_results)

## 2. TextRank Baseline
Thuật toán tóm tắt dựa trên đồ thị (Graph-based).

In [ ]:
def textrank_summarize(text, num_sentences=3):
    try:
        parser = PlaintextParser.from_string(text, Tokenizer("english")) # sumy tokenizer
        summarizer = TextRankSummarizer()
        summary = summarizer(parser.document, num_sentences)
        return " ".join([str(sentence) for sentence in summary])
    except:
        # Fallback in case of empty parsing
        return lead_3_summarize(text)

print("\nRunning TextRank...")
textrank_preds = [textrank_summarize(article) for article in articles]
textrank_results = evaluate_summaries(textrank_preds, references)
print("=== TextRank Metrics ===")
print(textrank_results)

## 3. Lưu kết quả Baseline
Phục vụ cho bước Error Analysis ở Report

In [ ]:
baseline_results = pd.DataFrame({
    "article": articles,
    "Reference": references,
    "Lead3": lead3_preds,
    "TextRank": textrank_preds
})
baseline_results.to_csv("baseline_predictions.csv", index=False)
print("Đã lưu baseline_predictions.csv thành công!")